<a href="https://colab.research.google.com/github/bps-rajora/Misinformation-detector/blob/main/NLPmultilingual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.9 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17396 sha256=055c8cbedbfe9d30bfc400ca3db60d57eb06794b51dcf79caf291d3c8acc6679
  Stored in directory: /root/.cache/pip/wheels/95/0f/04/b17a72024b56a60e499ce1a6313d283ed5ba332407155bee03
Successfully built googletrans
  Attempting uninstall: hyperframe
    Found existing installation: hyperframe 6.1.0
    Uninstalling hyperfra

In [32]:
!pip install pandas scikit-learn nltk scipy

In [33]:
from googletrans import Translator

translator = Translator()

In [34]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from scipy.sparse import hstack

In [35]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_table(url, header=None, names=['label','text'])

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [36]:
df['label'] = df['label'].map({'ham':0, 'spam':1})

df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [37]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [38]:
def count_caps(text):
    return sum(1 for c in text if c.isupper())

def count_links(text):
    return text.count("http")

def count_exclamations(text):
    return text.count("!")

In [39]:
df['caps'] = df['text'].apply(count_caps)
df['links'] = df['text'].apply(count_links)
df['exclam'] = df['text'].apply(count_exclamations)

df.head()

,label,text,clean_text,caps,links,exclam
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,3,0,0
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni,2,0,0
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final...,10,0,0
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,2,0,0
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,2,0,0


In [40]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)

X_text = vectorizer.fit_transform(df['clean_text'])

In [41]:
X_extra = df[['caps','links','exclam']].values

X = hstack([X_text, X_extra])

y = df['label']

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [43]:
model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

In [44]:
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.968609865470852


In [45]:
def translate_to_english(text):

    translated = translator.translate(text, dest='en')

    return translated.text

In [53]:
def predict_message(text):

    try:
        # Translate message to English
        translated_text = translate_to_english(text)
    except:
        # If translation fails, use original text
        translated_text = text

    # Clean translated text
    clean = clean_text(translated_text)

    vec = vectorizer.transform([clean])

    caps = count_caps(translated_text)
    links = count_links(translated_text)
    exclam = count_exclamations(translated_text)

    extra = np.array([[caps,links,exclam]])

    final = hstack([vec, extra])

    pred = model.predict(final)

    if pred[0] == 1:
        return "Spam / Possible misinformation"
    else:
        return "Normal message"

In [54]:
predict_message("Congratulations! You've been selected for a free vacation. Click here to claim!")

'Spam / Possible misinformation'

In [48]:
predict_message("WIN FREE BITCOIN NOW!!! CLICK HERE!!!")

'Spam / Possible misinformation'

In [49]:
predict_message("you won a 1k dollar")

'Normal message'

In [50]:
predict_message("URGENT! You've won a FREE iPhone! Claim now at this link: http://fakeurl.com")

'Spam / Possible misinformation'

Let's test the model with another message:

In [57]:
predict_message("अभी फ्री रिचार्ज पाओ क्लिक करो")

'Normal message'

In [58]:
predict_message("बधाई हो! आपने 1,000,000 रुपये जीते हैं। इस लिंक पर क्लिक करें: http://fakeshoplive.com")

'Spam / Possible misinformation'

In [55]:
predict_message("आपका बैंक अकाउंट बंद हो जाएगा अभी लिंक खोलें")

'Normal message'